# End-to-end test — atomic notes with co-located audits

Feature under test (branch `dpg/atomic-notes-audit`): every archive `usage_note` / `Schema.notes` entry is now an atomic `Note(id, text, audit)` whose **`Audit`** (a live probe or a `manual` marker) sits on the same object. `evals/audit.py` derives a live regression runner straight from the archives' notes, so a stale probe names the **exact note** to fix.

This notebook drives the **real** MCP server (in-memory `fastmcp.Client`) and the **real** audit runner (live TAP probes against NOIRLab Data Lab). Four checks:

1. The MCP tool envelope is unchanged — notes render as `list[str]`, no `Audit` leaks.
2. Every note carries exactly one co-located audit (the 1-1 link).
3. The derived runner live-probes each note and reports a per-note verdict.
4. A drifted note is flagged **STALE** with its exact `archives/<x>.py :: <id>` address.

In [ ]:
# Make the repo root importable so `evals` (a top-level package, not installed)
# resolves no matter where Jupyter is launched from inside the repo.
import pathlib
import sys

for _cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_cand / "pyproject.toml").exists() and (_cand / "evals").is_dir():
        sys.path.insert(0, str(_cand))
        print("repo root:", _cand)
        break

In [ ]:
from astro_archives_mcp.archives import get_active_archives
from astro_archives_mcp.archives._model import Note

archives = get_active_archives()
print("Active archives:", [a.short_name for a in archives])

## 1 · The MCP server surfaces atomic notes as plain strings (envelope unchanged)

We boot the real server and call `vo_archive_list` and `vo_schema_describe` exactly as an LLM client would. The `Audit` metadata must stay internal — the tool output is still a `list[str]`.

In [ ]:
from fastmcp import Client

from astro_archives_mcp.app import build_mcp

mcp = build_mcp()
async with Client(mcp) as client:
    arch = (await client.call_tool("vo_archive_list", {"short_name": "datalab"})).structured_content
    sch = (
        await client.call_tool(
            "vo_schema_describe", {"archive": "nrao", "table": "tap_schema.obscore"}
        )
    ).structured_content

entry = arch["archives"][0]
notes = entry["usage_notes"]
print(
    "vo_archive_list -> datalab.usage_notes:",
    len(notes),
    "notes; all str?",
    all(isinstance(n, str) for n in notes),
)
print("  audit key leaked into archive entry?", "audit" in entry)
print("  first note:", notes[0][:90], "...")
print()
print(
    "vo_schema_describe -> nrao/tap_schema.obscore notes:",
    len(sch["notes"]),
    "; all str?",
    all(isinstance(n, str) for n in sch["notes"]),
)
print("  audit key leaked into schema payload?", "audit" in sch)

assert all(isinstance(n, str) for n in notes)
assert all(isinstance(n, str) for n in sch["notes"])
assert "audit" not in entry and "audit" not in sch
print("\nPASS: notes render as list[str]; Audit never leaks into an envelope.")

## 2 · Every note carries exactly one co-located audit (the 1-1 link)

`collect_audits` walks the active archives and yields `(archive, note)` for every usage_note + schema note. Each note has one `Audit` — a probe (`ok/error/empty/nonempty/count`) or `manual`.

In [ ]:
from collections import Counter

from evals.audit import collect_audits

pairs = collect_audits(archives)
probeable = sum(1 for _, n in pairs if n.audit.expect != "manual")
manual = sum(1 for _, n in pairs if n.audit.expect == "manual")
print(f"{len(pairs)} atomic notes total  ->  {probeable} probeable, {manual} manual")
print()
per_archive = Counter(a.short_name for a, _ in pairs)
for name, count in per_archive.items():
    print(f"  {name:9s} {count} notes")
print()
# show one probeable note with its co-located audit
a0, n0 = next(
    (a, n)
    for a, n in pairs
    if a.short_name == "datalab" and n.id == "geometry-contains-untranslated"
)
print(f"sample note  archives/{a0.short_name}.py :: {n0.id}")
print("  text  :", n0.text[:100], "...")
print("  audit :", n0.audit.expect, "|", n0.audit.adql[:70], "...")

# the coverage invariant: a Note cannot be constructed without an Audit
try:
    Note(id="x", text="t", audit=None)
except TypeError as e:
    print("\nconstruction gate: Note(audit=None) ->", type(e).__name__)

## 3 · The derived runner live-probes each note (real network → NOIRLab Data Lab)

`audit.run([datalab])` runs a liveness control probe, then each note's audit against the real TAP service, and returns a per-note verdict keyed to `archives/datalab.py :: <id>`.

In [ ]:
from IPython.display import HTML, display

from evals import audit

datalab = [a for a in archives if a.short_name == "datalab"]
try:
    rows = audit.run(datalab)
except Exception as e:  # network hiccup shouldn't kill the notebook
    rows = []
    print("live run raised:", type(e).__name__, e)


def status_color(s):
    return {
        "still_true": "#1a7f37",
        "stale": "#b91c1c",
        "unreachable": "#9a6700",
        "manual": "#6e7781",
    }.get(s, "#000")


html = [
    '<table style="border-collapse:collapse;font-family:monospace;font-size:12px">',
    '<tr><th style="text-align:left;padding:2px 10px">status</th>'
    '<th style="text-align:left;padding:2px 10px">note_id</th>'
    '<th style="text-align:left;padding:2px 10px">source</th></tr>',
]
for r in rows:
    sc = status_color(r["status"])
    html.append(
        f'<tr><td style="padding:2px 10px;color:{sc};font-weight:bold">{r["status"].upper()}</td>'
        f'<td style="padding:2px 10px">{r["note_id"]}</td>'
        f'<td style="padding:2px 10px;color:#57606a">{r["source"]}</td></tr>'
    )
html.append("</table>")
display(HTML("".join(html)))

counts = Counter(r["status"] for r in rows)
print("\nverdict summary:", dict(counts))
print("STALE notes:", [r["note_id"] for r in rows if r["status"] == "stale"] or "none")

## 4 · Failure → fix targeting (the headline)

The point of the feature: **when an audit fails, we know exactly which note is outdated.** We simulate a drift — pretend Data Lab started *translating* the ADQL `CONTAINS` geometry (so the note `geometry-contains-untranslated`, which expects an **error**, now sees a success). The runner must flag that one note **STALE** and print its exact source address.

In [ ]:
note = next(n for n in datalab[0].usage_notes if n.id == "geometry-contains-untranslated")
print("note claims:", note.text[:80], "...")
print("audit expects:", note.audit.expect, "(a probe that should ERROR)")
print()

# monkeypatch the probe layer to simulate the archive no longer erroring
orig_probe = audit._probe
audit._probe = lambda *a, **k: ("ok", 1, ["ra"], "")  # pretend CONTAINS now succeeds
try:
    drifted = audit.check_note(datalab[0], note, control_ok=True)
finally:
    audit._probe = orig_probe

print("verdict :", drifted["status"].upper())
print("fix here:", drifted["source"])
assert drifted["status"] == "stale"
assert drifted["source"] == "archives/datalab.py :: geometry-contains-untranslated"
print("\nPASS: a drifted note is flagged STALE and pinned to its exact archive/id address.")

## Result

End-to-end, through the real server and the real audit runner:

- ✅ MCP envelopes unchanged — atomic notes render as `list[str]`, `Audit` stays internal.
- ✅ Every note carries one co-located audit; a `Note` can't exist without one.
- ✅ The derived runner live-probes each note against the real archive.
- ✅ A drifted claim is flagged STALE and pinned to `archives/<archive>.py :: <note_id>` — the direct evals-audit ↔ archive-knowledge link the feature set out to deliver.